In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,to_date

In [0]:
spark = SparkSession.builder.appName("Silver Layer").getOrCreate()



In [0]:
df = spark.table("workspace.default.bronze_sales")

#df.head(10)
df.count()
--9994



In [0]:
#clean duplicates

df_Clean = df.dropDuplicates()

df_Clean.count() --9994

In [0]:
df_Clean = df.dropna()
df_Clean.count() --9994

In [0]:
df_Clean = df_Clean.withColumn("Order Date", to_date(col("Order Date"), "MM-dd-yyyy"))
df_Clean.count() #10003

In [0]:
df_Clean = df_Clean.filter(col("sales").isNotNull())

df_Clean = df_Clean.filter(col("quantity").cast("double") > 0)

df_Clean.count() #10003

In [0]:
#save to Unity Catalog table
df_Clean.write.mode("overwrite").option("delta.columnMapping.mode", "name").saveAsTable("silver_sales")

print("table created at silver layer")

In [0]:
%sql
select count(*) from 